# Groundy — Point Inspector

**Pick any coordinate on Earth and see every layer of data Groundy has for it:**
climate, satellite vegetation (NDVI/EVI), land-surface temperature, soil moisture,
rainfall, terrain, best-fit crops and recommended rotations.

Two ways to choose a point:
1. **Type coordinates** and run the inspect cell (always works).
2. **Click anywhere** on the interactive world map, then run the inspect cell.


## 0. Setup


In [4]:
# First run only: # %pip install -q geemap earthengine-api pandas matplotlib requests
import requests, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110
API = 'https://api.groundy.site'
print('Ready. API:', API)


Ready. API: https://api.groundy.site


## 1. The inspector function

`inspect_point(lat, lon)` queries the Groundy API and shows **everything** for that point.
First call for a new point takes ~60 s (live satellite query); repeats are instant (cache).


In [5]:
MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

def inspect_point(lat, lon):
    print(f'Querying {lat:.4f}, {lon:.4f} ...  (first time ~60s)')
    r = requests.post(API+'/analizar/punto', json={'lat':lat,'lon':lon}, timeout=200)
    if r.status_code != 200:
        print('Error', r.status_code, r.text[:200]); return None
    d = r.json()

    # ---- summary ----
    cl = d['climate']
    print('\n============================================================')
    print(f"  POINT  {lat:.4f}, {lon:.4f}   |   elevation {d['elev_m']} m")
    print(f"  Confidence: {d['confidence']}   |   closest reference: {d['sister_site']}")
    print('============================================================')
    print(f"  Mean temperature : {cl['tmean_c']} C")
    print(f"  Rainfall (POWER) : {cl['rain_mm_yr']} mm/yr   |  IMERG: {cl['rain_imerg_mm_yr']} mm/yr")
    print(f"  Rainfall range   : {cl['rain_range_mm']} mm/yr")
    print(f"  ET0 / aridity    : {cl['et0_mm_yr']} mm/yr  /  {cl['aridity']}")

    # ---- all monthly satellite layers ----
    vm = d['vegetation_monthly']; st = d['vegetation_status']
    panels = [('ndvi','NDVI (MODIS)','#1a9641'), ('evi','EVI (MODIS)','#66bd63'),
              ('lst_c','Land-surface temp (C)','#d73027'), ('sm_surf','Soil moisture surface','#4575b4'),
              ('sm_root','Soil moisture root zone','#313695'), ('rain_imerg','Rainfall IMERG (mm)','#74add1')]
    fig, axes = plt.subplots(2, 3, figsize=(15,7))
    for ax,(key,title,color) in zip(axes.ravel(), panels):
        vals = vm.get(key)
        if vals and st.get(key)=='ok':
            if key=='rain_imerg': ax.bar(MONTHS, vals, color=color)
            else: ax.plot(MONTHS, vals, '-o', color=color)
        else:
            ax.text(0.5,0.5,'no data',ha='center',va='center',transform=ax.transAxes,color='gray')
        ax.set_title(title, fontsize=10); ax.tick_params(axis='x',rotation=45,labelsize=7); ax.grid(alpha=.3)
    fig.suptitle(f'Monthly satellite data — {lat:.3f}, {lon:.3f}', fontsize=13, weight='bold')
    plt.tight_layout(); plt.show()

    # ---- crops & rotations ----
    print('\n  TOP CROPS for this point:')
    crops = pd.DataFrame(d['crops_top'])[['id','suit','best_month','limits']]
    display(crops)
    print('  RECOMMENDED ROTATIONS:')
    rot = pd.DataFrame(d['rotations'])[['name','score']].rename(columns={'name':'Rotation','score':'Score'})
    display(rot)
    print('  Best rotation:', d['best_rotation'])
    print('\n  Data sources:', ', '.join(f"{k}={v}" for k,v in d['provenance'].items() if k!='note'))
    return d


## 2. Option A — type any coordinate

Change the numbers to any place on Earth and run the cell.


In [ ]:
# Examples:  Pasto 1.21,-77.28 · Iowa 41.6,-93.61 · Punjab 30.9,75.85 · Niger 13.51,2.11
lat, lon = 1.21, -77.28
data = inspect_point(lat, lon)


## 3. Option B — click anywhere on the world map

Run this cell, then **click** on the map. The chosen coordinate is printed and stored;
then run the cell in section 4.


In [ ]:
import ee, geemap
from pathlib import Path
try:
    cred = ee.ServiceAccountCredentials('papanauta-server@kinetic-anvil-457906-e8.iam.gserviceaccount.com',
                                        str(Path('../secrets/gee_key.json')))
    ee.Initialize(cred, project='kinetic-anvil-457906-e8')
except Exception:
    ee.Authenticate(); ee.Initialize(project='kinetic-anvil-457906-e8')

# optional background layer: global NDVI
ndvi = (ee.ImageCollection('MODIS/061/MOD13Q1').select('NDVI')
        .filterDate('2023-01-01','2025-12-31').mean().multiply(0.0001))
green = {'min':0,'max':0.8,'palette':['#d7191c','#fdae61','#ffffbf','#a6d96a','#1a9641']}

selected = {'lat':None,'lon':None}
Map = geemap.Map(center=[10,0], zoom=2)
Map.addLayer(ndvi, green, 'Global NDVI')
def _on_click(**kw):
    if kw.get('type')=='click':
        la, lo = kw['coordinates']
        selected['lat'], selected['lon'] = round(la,4), round(lo,4)
        print('Selected point:', selected['lat'], selected['lon'], '-> run section 4')
Map.on_interaction(_on_click)
Map


## 4. Inspect the clicked point


In [ ]:
assert selected['lat'] is not None, 'Click a point on the map in section 3 first.'
data = inspect_point(selected['lat'], selected['lon'])


## What you just saw

For a single coordinate anywhere on Earth: full climate, six monthly satellite layers
(NDVI, EVI, land-surface temperature, surface & root-zone soil moisture, rainfall),
terrain, 42-crop suitability and recommended rotations — all from one API call.

Sources: NASA POWER · GPM IMERG · MODIS · SMAP · Sentinel-2 · SRTM · FAO EcoCrop.
